<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-06 22:33:10
=================== PORTFOLIO QUALITY ====================
qualified stocks: 85
with latest results: 85
still star stocks: 45
=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.53 C
CAGR/XIRR %:     1.22%
Est FTT Value:   2.74 C
Future Upside:   1.21 C (78.67%)
=================== PERFORMANCE ==========================
Current PnL:             -43.76 L (-22.8%)
Total Booked PnL:        49.23 L (33.02%)
Booked + Current PnL:    5.46 L (3.66%)
CY Booked PnL:           6.36 L (4.14%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               51.02 K (0.33%)
CY Investment:            1.92 C
Reserve:                  2.74 L
=================== P&L DETAIL ============================
Total Profit:             1.70 L
Total Loss:               -45.47 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
76,TTKPRESTIG,10.47,-20.87,26.49,0.09,21125.0,-21031.0,79746.0,770.00,-3.77,73.0,M-SC,3.48,250.0,-1.00,0.53,43.53,OX40N,NTT,DURABLES
40,ICICIPRULI,1.09,-29.23,22.14,-13.56,30866.0,-57591.0,139414.0,522.35,-54.58,44.0,X-MC,9.52,78.0,-1.87,0.93,5.07,X40,ATH,INSURANCE
12,BOSCH-HCIL,0.68,-25.89,34.93,-0.01,33083.0,-33092.0,94713.0,2282.00,737.99,44.0,M-SC,19.88,231.0,-1.00,0.63,61.21,OX40N,BTT,AC
66,SIS,-1.99,-12.33,38.31,21.26,37293.0,-13686.0,97346.0,528.00,86.99,44.0,H-SC,7.89,168.0,-0.37,0.65,44.74,OX40N,NTT,MISC
77,UNITDSPR,1.69,4.17,15.94,20.77,38877.0,9752.0,243896.0,1644.00,-11.27,59.0,X-MC,3.50,64.0,0.25,1.62,17.24,X40N,NTT,BREWERIES


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
38,HYUNDAI,0.42,10.17,30.64,43.94,84842.0,25568.0,276900.0,2782.73,14.27,66.0,X-LC,5.09,29.0,0.30,1.84,27.13,X40N,ATH,AUTO
52,MARUTI,0.77,-6.85,34.27,25.08,96127.0,-20618.0,280500.0,17120.06,-12.53,71.0,X-LC,12.95,33.0,-0.21,1.86,14.15,XR,ATH,AUTO
23,DMART,0.15,-15.87,57.41,32.43,171067.0,-56225.0,297975.0,5391.45,-30.04,52.0,X-LC,16.35,14.0,-0.33,1.98,9.29,X40N,ATH,FMCG
35,HDFCLIFE,3.95,-2.08,43.54,40.56,132665.0,-6462.0,304697.0,811.41,-28.77,65.0,X-LC,9.76,31.0,-0.05,2.02,10.07,X40,ATH,INSURANCE
4,ASIANPAINT,0.88,-7.71,34.42,24.06,105160.0,-25512.0,305520.0,3422.40,-19.19,69.0,X-LC,5.82,30.0,-0.24,2.03,21.35,X40,ATH,PAINTS


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
76,TTKPRESTIG,10.47,-20.87,26.49,0.09,21125.0,-21031.0,79746.0,770.00,-3.77,73.0,M-SC,3.48,250.0,-1.00,0.53,43.53,OX40N,NTT,DURABLES
32,HAVELLS,5.61,-27.78,81.97,31.42,194171.0,-91111.0,236880.0,2052.62,-37.81,62.0,X-MC,12.97,63.0,-0.47,1.57,11.85,X40,ATH,ELECTRICAL
51,M&M,5.11,2.63,14.47,17.48,44712.0,7922.0,308997.0,3762.88,-10.60,76.0,X-LC,8.40,25.0,0.18,2.05,19.48,X40N,ATH,AUTO
67,STARHEALTH,4.13,9.28,34.75,47.26,106760.0,26101.0,307224.0,761.00,-3.55,59.0,H-SC,4.80,173.0,0.24,2.04,32.84,XY24,NTT,INSURANCE
18,COFFEEDAY,3.96,-46.77,154.13,35.27,93159.0,-53107.0,60442.0,80.00,NaN,61.0,NaN,1.52,266.0,-0.57,0.40,49.05,XR,NTT,HOTELS


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-7.09,38.18,49.40,106.44,100784.0,56373.0,204017.0,1629.94,0.92,34.0,M-SC,24.99,220.0,0.56,1.35,54.58,OX40N,ATH,CABLES
11,BERGEPAINT,-5.28,-11.85,42.16,25.32,95794.0,-30533.0,227216.0,680.00,-23.28,53.0,X-MC,2.03,68.0,-0.32,1.51,19.15,XY24,NTT,PAINTS
80,VALIANTORG,-5.13,-27.18,139.16,74.15,165790.0,-44469.0,119136.0,838.00,NaN,35.0,NaN,44.01,145.0,-0.27,0.79,77.29,XR,NTT,CHEMICALS
50,LAOPALA,-4.89,-55.49,212.52,39.09,124003.0,-72756.0,58349.0,464.00,NaN,47.0,NaN,15.06,154.0,-0.59,0.39,6.79,AR,NTT,CERAMICS
49,KANSAINER,-4.03,-34.58,73.46,13.47,129590.0,-93258.0,176409.0,340.00,-31.54,59.0,H-SC,7.22,162.0,-0.72,1.17,24.69,XY24,NTT,PAINTS


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-7.09,38.18,49.40,106.44,100784.0,56373.0,204017.0,1629.94,0.92,34.0,M-SC,24.99,220.0,0.56,1.35,54.58,OX40N,ATH,CABLES
66,SIS,-1.99,-12.33,38.31,21.26,37293.0,-13686.0,97346.0,528.00,86.99,44.0,H-SC,7.89,168.0,-0.37,0.65,44.74,OX40N,NTT,MISC
76,TTKPRESTIG,10.47,-20.87,26.49,0.09,21125.0,-21031.0,79746.0,770.00,-3.77,73.0,M-SC,3.48,250.0,-1.00,0.53,43.53,OX40N,NTT,DURABLES
12,BOSCH-HCIL,0.68,-25.89,34.93,-0.01,33083.0,-33092.0,94713.0,2282.00,737.99,44.0,M-SC,19.88,231.0,-1.00,0.63,61.21,OX40N,BTT,AC
49,KANSAINER,-4.03,-34.58,73.46,13.47,129590.0,-93258.0,176409.0,340.00,-31.54,59.0,H-SC,7.22,162.0,-0.72,1.17,24.69,XY24,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-7.09,38.18,49.4,106.44,100784.0,56373.0,204017.0,1629.94,0.92,34.0,M-SC,24.99,220.0,0.56,1.35,54.58,OX40N,ATH,CABLES


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-7.09,38.18,49.40,106.44,100784.0,56373.0,204017.0,1629.94,0.92,34.0,M-SC,24.99,220.0,0.56,1.35,54.58,OX40N,ATH,CABLES
67,STARHEALTH,4.13,9.28,34.75,47.26,106760.0,26101.0,307224.0,761.00,-3.55,59.0,H-SC,4.80,173.0,0.24,2.04,32.84,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-7.09,38.18,49.40,106.44,100784.0,56373.0,204017.0,1629.94,0.92,34.0,M-SC,24.99,220.0,0.56,1.35,54.58,OX40N,ATH,CABLES
84,WIPRO,-0.20,-9.01,112.12,93.01,255297.0,-22540.0,227700.0,420.00,-23.12,68.0,M-LC,7.84,98.0,-0.09,1.51,26.28,XR,NTT,IT
70,TANLA,-0.65,-26.37,192.24,115.17,383248.0,-71402.0,199359.0,1903.93,NaN,74.0,NaN,14.19,147.0,-0.19,1.32,76.23,AR,ATH,IT
42,INDIAMART,-1.85,-33.52,202.22,100.92,165812.0,-41340.0,81996.0,4675.60,NaN,44.0,NaN,16.72,140.0,-0.25,0.54,4.03,AR,ATH,MISC
3,ALKYLAMINE,0.30,-33.02,187.46,92.54,126772.0,-33337.0,67626.0,4520.85,NaN,34.0,NaN,8.12,142.0,-0.26,0.45,28.42,SR,ATH,CHEMICALS


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
37,HONAUT,-0.79,-22.01,93.46,50.88,140564.0,-42438.0,150400.0,58192.80,-39.00,36.0,X-SC,6.99,90.0,-0.30,1.00,14.59,X40N,ATH,ELECTRICAL
6,AWL,-0.48,-42.49,190.42,67.02,390188.0,-151400.0,204909.0,485.00,-57.56,38.0,X-SC,7.90,81.0,-0.39,1.36,1.92,XY24,NTT,FMCG
40,ICICIPRULI,1.09,-29.23,22.14,-13.56,30866.0,-57591.0,139414.0,522.35,-54.58,44.0,X-MC,9.52,78.0,-1.87,0.93,5.07,X40,ATH,INSURANCE
27,GLAXO,0.98,1.35,33.88,35.69,68432.0,2700.0,201984.0,3380.16,-24.10,45.0,X-MC,9.33,56.0,0.04,1.34,20.05,X40,ATH,PHARMA
7,BAJAJHFL,-3.80,-28.54,126.88,62.13,247568.0,-77926.0,195120.0,181.50,-40.31,47.0,X-MC,5.32,60.0,-0.31,1.29,9.45,X40N,ATH,FINANCE


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
1,ABBOTINDIA,-0.98,-11.62,30.54,15.37,48275.0,-20785.0,158070.0,34390.26,-27.99,53.0,X-MC,1.16,58.0,-0.43,1.05,7.07,X40,ATH,PHARMA
17,CIPLA,0.49,6.06,16.81,23.89,53605.0,18231.0,318886.0,1648.14,20.33,67.0,X-LC,1.19,18.0,0.34,2.12,19.44,X40N,ATH,PHARMA
11,BERGEPAINT,-5.28,-11.85,42.16,25.32,95794.0,-30533.0,227216.0,680.00,-23.28,53.0,X-MC,2.03,68.0,-0.32,1.51,19.15,XY24,NTT,PAINTS
55,PAGEIND,1.15,-6.29,41.76,32.85,105031.0,-16869.0,251510.0,50936.07,-30.28,53.0,X-MC,2.06,55.0,-0.16,1.67,20.10,X40,ATH,APPARELS
77,UNITDSPR,1.69,4.17,15.94,20.77,38877.0,9752.0,243896.0,1644.00,-11.27,59.0,X-MC,3.50,64.0,0.25,1.62,17.24,X40N,NTT,BREWERIES


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
6,AWL,-0.48,-42.49,190.42,67.02,390188.0,-151400.0,204909.0,485.00,-57.56,38.0,X-SC,7.90,81.0,-0.39,1.36,1.92,XY24,NTT,FMCG
28,GODREJCP,-1.36,-22.59,70.64,32.10,136460.0,-56360.0,193177.0,1452.16,-24.07,51.0,X-MC,16.50,32.0,-0.41,1.28,4.67,X40N,ATH,FMCG
40,ICICIPRULI,1.09,-29.23,22.14,-13.56,30866.0,-57591.0,139414.0,522.35,-54.58,44.0,X-MC,9.52,78.0,-1.87,0.93,5.07,X40,ATH,INSURANCE
56,PGHH,-0.66,-47.68,150.42,31.02,168037.0,-101808.0,111712.0,17484.41,-61.49,48.0,X-SC,24.11,57.0,-0.61,0.74,5.17,X40,ATH,FMCG
20,DABUR,-1.21,-25.64,93.24,43.70,172000.0,-63598.0,184470.0,735.00,-35.86,51.0,X-MC,12.63,70.0,-0.37,1.22,6.30,XY24,BTT,FMCG


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
56,PGHH,-0.66,-47.68,150.42,31.02,168037.0,-101808.0,111712.0,17484.41,-61.49,48.0,X-SC,24.11,57.0,-0.61,0.74,5.17,X40,ATH,FMCG
40,ICICIPRULI,1.09,-29.23,22.14,-13.56,30866.0,-57591.0,139414.0,522.35,-54.58,44.0,X-MC,9.52,78.0,-1.87,0.93,5.07,X40,ATH,INSURANCE
2,ACC,0.77,-44.67,215.97,74.81,320379.0,-119784.0,148344.0,3906.00,-44.95,62.0,X-SC,15.43,85.0,-0.37,0.98,9.89,XY24,BTT,CEMENT
37,HONAUT,-0.79,-22.01,93.46,50.88,140564.0,-42438.0,150400.0,58192.80,-39.00,36.0,X-SC,6.99,90.0,-0.30,1.00,14.59,X40N,ATH,ELECTRICAL
1,ABBOTINDIA,-0.98,-11.62,30.54,15.37,48275.0,-20785.0,158070.0,34390.26,-27.99,53.0,X-MC,1.16,58.0,-0.43,1.05,7.07,X40,ATH,PHARMA


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
47,ITC,1.54,-19.38,59.63,28.70,210378.0,-84790.0,352805.0,452.00,-27.09,69.0,X-LC,7.48,1.0,-0.40,2.34,11.04,X40,NTT,FMCG
72,TCS,0.11,-24.75,68.43,26.75,251330.0,-120795.0,367280.0,4208.34,-38.95,63.0,X-LC,6.41,2.0,-0.48,2.44,27.39,X40,ATH,IT
45,INFY,1.20,-14.53,66.92,42.66,229272.0,-58249.0,342606.0,1972.00,-38.13,64.0,X-LC,8.44,3.0,-0.25,2.27,22.65,X40,NTT,IT
74,TMPV,-0.58,-28.92,105.48,46.05,335414.0,-129384.0,317988.0,600.00,-85.99,60.0,X-LC,17.22,4.0,-0.39,2.11,11.79,XY24,NTT,AUTO
33,HCLTECH,0.66,-1.47,27.57,25.70,98061.0,-5293.0,355680.0,1837.07,-6.32,67.0,X-LC,6.97,8.0,-0.05,2.36,41.91,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
76,TTKPRESTIG,10.47,-20.87,26.49,0.09,21125.0,-21031.0,79746.0,770.00,-3.77,73.0,M-SC,3.48,250.0,-1.00,0.53,43.53,OX40N,NTT,DURABLES
12,BOSCH-HCIL,0.68,-25.89,34.93,-0.01,33083.0,-33092.0,94713.0,2282.00,737.99,44.0,M-SC,19.88,231.0,-1.00,0.63,61.21,OX40N,BTT,AC
80,VALIANTORG,-5.13,-27.18,139.16,74.15,165790.0,-44469.0,119136.0,838.00,NaN,35.0,NaN,44.01,145.0,-0.27,0.79,77.29,XR,NTT,CHEMICALS
70,TANLA,-0.65,-26.37,192.24,115.17,383248.0,-71402.0,199359.0,1903.93,NaN,74.0,NaN,14.19,147.0,-0.19,1.32,76.23,AR,ATH,IT
38,HYUNDAI,0.42,10.17,30.64,43.94,84842.0,25568.0,276900.0,2782.73,14.27,66.0,X-LC,5.09,29.0,0.30,1.84,27.13,X40N,ATH,AUTO


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
70,TANLA,-0.65,-26.37,192.24,115.17,383248.0,-71402.0,199359.0,1903.93,NaN,74.0,NaN,14.19,147.0,-0.19,1.32,76.23,AR,ATH,IT
71,TATAELXSI,-1.66,-44.44,137.64,32.03,100814.0,-58590.0,73245.0,9161.00,-36.88,63.0,H-SC,13.55,158.0,-0.58,0.49,28.67,OX40N,NTT,IT
84,WIPRO,-0.20,-9.01,112.12,93.01,255297.0,-22540.0,227700.0,420.00,-23.12,68.0,M-LC,7.84,98.0,-0.09,1.51,26.28,XR,NTT,IT
13,BSOFT,2.13,-38.84,140.67,47.20,121010.0,-54629.0,86024.0,811.90,NaN,64.0,NaN,11.07,172.0,-0.45,0.57,26.96,XR,ATH,IT
33,HCLTECH,0.66,-1.47,27.57,25.70,98061.0,-5293.0,355680.0,1837.07,-6.32,67.0,X-LC,6.97,8.0,-0.05,2.36,41.91,X40,ATH,IT


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.73
1,25,50.78
2,50,81.25


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    42.24
MC    20.84
SC    20.39
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      28.16
XY24     22.72
X40N     19.04
XR        9.84
XY25      9.65
OX40N     6.61
AR        3.23
SR        0.75
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    40.73
X-MC    18.43
H-SC     8.28
X-SC     7.11
M-SC     5.00
M-LC     1.51
M-MC     1.08
H-MC     0.99
L-MC     0.34
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,18.03,-26.20,71.50
IT,13.18,-29.07,96.32
FINANCE,10.77,-24.18,77.22
AUTO,8.76,-13.47,53.95
INSURANCE,6.15,-6.83,35.89
MISC,5.48,-49.63,109.77
PAINTS,5.46,-25.59,47.63
BANKS,5.17,-15.00,72.67
PHARMA,4.51,0.02,25.08


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,4012125.0,19
X40,2146709.0,17
XR,1569232.0,13
X40N,1468299.0,12
AR,1020715.0,6
OX40N,795476.0,10
XY25,758813.0,6
SR,300420.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,2804568.0,19
X-MC,1627562.0,14
X-SC,1202258.0,6
H-SC,905776.0,8
M-SC,407754.0,6
H-MC,264711.0,1
M-LC,255297.0,1
M-MC,82150.0,1
L-MC,50717.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,,sum,count
Conviction,Criteria,,
X-LC,X40,1244258.0,8
X-SC,XY24,710567.0,2
X-LC,XY24,555411.0,2
H-SC,XY24,551241.0,3
X-MC,X40N,542459.0,4
X-LC,X40N,540556.0,5
X-MC,X40,508862.0,6
X-LC,XY25,368216.0,3
X-MC,XY25,308447.0,2


In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 44.0 seconds
